In [15]:
%pip install pandas quinex click quinex-utils tdqm numpy==1.25

INFO: pip is looking at multiple versions of quinex-utils to determine which version is compatible with other requirements. This could take a while.
  Using cached quinex-0.0.0-py3-none-any.whl.metadata (17 kB)
INFO: pip is looking at multiple versions of pandas to determine which version is compatible with other requirements. This could take a while.
  Using cached pandas-3.1.0-cp311-cp311-win_amd64.whl.metadata (18 kB)
INFO: pip is still looking at multiple versions of pandas to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
  Using cached pandas-2.3.1-cp311-cp311-win_amd64.whl.metadata (19 kB)
INFO: pip is still looking at multiple versions of quinex-utils to determine which version is compatible with oth

  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [21 lines of output]
      Traceback (most recent call last):
        File "c:\Users\3265458R\OneDrive - University of Glasgow\Documents\african_species\.venv\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 389, in <module>
          main()
        File "c:\Users\3265458R\OneDrive - University of Glasgow\Documents\african_species\.venv\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        File "c:\Users\3265458R\OneDrive - University of Glasgow\Documents\african_species\.venv\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 143, in get_requires_for_build_wheel
          return hook(config_settings)
                 ^^^^^^^^^

In [10]:
!python3 -m spacy download en_core_web_md


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 189, in _run_module_as_main
  File "<frozen runpy>", line 148, in _get_module_details
  File "<frozen runpy>", line 112, in _get_module_details
  File "c:\Users\3265458R\OneDrive - University of Glasgow\Documents\african_species\.venv\Lib\site-packages\spacy\__init__.py", line 6, in <module>
    from .errors import setup_default_warnings
  File "c:\Users\3265458R\OneDrive - University of Glasgow\Documents\african_species\.venv\Lib\site-packages\spacy\errors.py", line 3, in <module>
    from .c

In [11]:
%pip install git+https://github.com/FZJ-IEK3-VSA/llm-chunker.git

  Cloning https://github.com/FZJ-IEK3-VSA/llm-chunker.git to C:\Users\3265458R\AppData\Local\Temp\pip-req-build-54cnubsw
  Resolved https://github.com/FZJ-IEK3-VSA/llm-chunker.git to commit 392e686387bbd781ec591dd73819ea3025cbebc0
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Note: you may need to restart the kernel to use updated packages.


  Running command git clone --filter=blob:none --quiet https://github.com/FZJ-IEK3-VSA/llm-chunker.git 'C:\Users\3265458R\AppData\Local\Temp\pip-req-build-54cnubsw'

[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
from quinex import Quinex
import pandas as pd
from tqdm import tqdm

DESCRIPTION_CSV = "cleaned_labeled_descriptions.csv"

description_df = pd.read_csv(DESCRIPTION_CSV)

quinex = Quinex()

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
import pandas as pd

# in case a key is missing in a nested dictionary, return None instead of raising an error
def safe_get(d, *keys):
    for key in keys:
        if not isinstance(d, dict):
            return None
        d = d.get(key)
    return d

import pandas as pd


def parse_quinex_results(results):
    # if Quinex returns no results, return an empty DataFrame
    if not results:
        return pd.DataFrame()

    rows = []

    for record in results:

        claim = record.get("claim") or {}

        entity = (claim.get("entity") or {}).get("text")
        property_name = (claim.get("property") or {}).get("text")

        quantity = claim.get("quantity") or {}

        quantity_text = quantity.get("text")

        normalized = quantity.get("normalized") or {}

        quantity_type = normalized.get("type")

        individual_quantities = (
            (normalized.get("individual_quantities") or {})
            .get("normalized", [])
        )

        values = []
        unit = None

        for q in individual_quantities:

            numeric_value = (
                ((q.get("value") or {})
                 .get("normalized") or {})
                .get("numeric_value")
            )

            if numeric_value is not None:
                values.append(numeric_value)

            if unit is None:
                unit_info = (q.get("unit") or {})
                normalized_units = unit_info.get("normalized") or []

                if normalized_units:
                    unit = normalized_units[0].get("text")
                else:
                    # fall back to raw unit text
                    text_info = unit_info.get("text") or {}

                    unit = (
                        text_info.get("suffixed")
                        or text_info.get("ellipsed")
                    )

        rows.append({
            "entity": entity,
            "property": property_name,
            "value_min": min(values) if values else None,
            "value_max": max(values) if values else None,
            "unit": unit,
            "quantity_text": quantity_text,
            "quantity_type": quantity_type,
        })

    return pd.DataFrame(rows)

In [ ]:
#Sanity check
text = description_df[description_df["detected_language"] == "ENGLISH"]["cleaned_description"].iloc[110]

qclaims = quinex(text)

print(text)
print(parse_quinex_results(qclaims))

In [ ]:
import os
import pickle
import pandas as pd
from tqdm import tqdm

CHECKPOINT_DATA = "quinex_results.pkl"
CHECKPOINT_STATE = "quinex_state.pkl"
SAVE_EVERY = 10

In [ ]:
if os.path.exists(CHECKPOINT_DATA) and os.path.exists(CHECKPOINT_STATE):

    result_df = pd.read_pickle(CHECKPOINT_DATA)

    with open(CHECKPOINT_STATE, "rb") as f:
        start_idx = pickle.load(f)

    print(f"Resuming from row {start_idx:,}")

else:
    result_df = pd.DataFrame()
    start_idx = 0

english_descriptions = (
    description_df[description_df["detected_language"] == "ENGLISH"]
    .reset_index(drop=True)
)

for idx in tqdm(
    range(start_idx, len(english_descriptions)),
    total=len(english_descriptions),
    initial=start_idx,
):

    row = english_descriptions.iloc[idx]

    try:
        qclaims = quinex(row.cleaned_description)

        parsed_df = parse_quinex_results(qclaims)

        if not parsed_df.empty:
            parsed_df["ipni_id"] = row.ipni_id
            parsed_df["provider"] = row.provider

            result_df = pd.concat(
                [result_df, parsed_df],
                ignore_index=True,
            )

    except Exception as e:
        print(f"Error at row {idx}: {e}")

    if (idx + 1) % SAVE_EVERY == 0:

        result_df.to_pickle(CHECKPOINT_DATA)

        with open(CHECKPOINT_STATE, "wb") as f:
            pickle.dump(idx + 1, f)

result_df.to_pickle(CHECKPOINT_DATA)

print(f"Final checkpoint data saved to {CHECKPOINT_DATA}")

with open(CHECKPOINT_STATE, "wb") as f:
    pickle.dump(len(english_descriptions), f)

print(f"Final checkpoint state saved to {CHECKPOINT_STATE}")

: 

In [ ]:
import pandas as pd

result_df = pd.read_pickle("quinex_results.pkl")

result_df.head()

result_df.to_csv("extracted_quantities.csv", index=False)